# Revanth — Combined Data Analysis and Ridge Regression

**Group 5: Multimodal Deep Learning for Reliable GRACE Water Storage Trends**

Your assigned contribution combines rainfall, temperature, and water storage, explores correlations and distributions, and evaluates Ridge against simple reference models.

**Status: real-data execution pending.** This notebook has no prefilled results. Get `aligned_0.5deg.nc` from Rachan’s notebook outputs. The supplied ZIP does not contain it. Read `reports/revanth_combined_ridge.md` for experimental choices and limitations.

## 1. Setup
Run from a local checkout, or clone this contribution’s branch into Colab first. In Colab, put the aligned dataset in Drive and mount Drive if needed. Set `REPO` and `INPUT` below to their actual locations. Installation only installs packages; it does not download NASA data.

In [ ]:
from pathlib import Path
import sys, subprocess
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
INPUT = REPO / "data" / "processed" / "aligned_0.5deg.nc"
OUTPUT = REPO / "results" / "revanth_combined_ridge"
assert (REPO / "scripts" / "combined_ridge.py").exists(), "Set REPO to your project checkout"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-revanth.txt")], check=True)

## 2. Inspect the combined handoff
All three measurements must describe the same places and calendar months. Rainfall is mm/month, temperature is Celsius, and TWS is equivalent water height in cm. The script checks this contract and restores missing calendar months as missing data.

In [ ]:
import xarray as xr
if not INPUT.is_file():
    raise FileNotFoundError(f"Missing {INPUT}. Obtain aligned_0.5deg.nc from Rachan and update INPUT.")
with xr.open_dataset(INPUT) as ds:
    display(ds)
    print("Source metadata:", ds.attrs)

## 3. Train, select, and evaluate
2019–2022 teaches the model; 2023 chooses how much to shrink its coefficients; 2024 onward tests it. Inputs stop at the previous calendar month. The chosen model is refitted on 2019–2023. Four models use exactly the same test observations.

This assumes last month's observations are available and does not model NASA publication delays. It is a sequence of one-month forecasts, not a two-year forecast made at once.

In [ ]:
sys.path.insert(0, str(REPO / "scripts"))
from combined_ridge import run
metrics = run(INPUT, REPO / "configs" / "grace_fo_config.yaml", OUTPUT)
display(metrics)

## 4. Correlations and distributions
Correlations describe how measurements move together; they do not prove cause. Removing the training seasonal cycle reduces relationships driven just by wet/dry or warm/cold seasons. Distribution plots show whether the range of values changes across years.

In [ ]:
import pandas as pd
from IPython.display import display, Image
for filename in ["correlations.csv", "distributions.csv", "alpha_selection.csv"]:
    print(filename)
    display(pd.read_csv(OUTPUT / filename))
for filename in ["distributions.png", "correlations.png", "test_predictions.png"]:
    display(Image(filename=str(OUTPUT / "figures" / filename)))

## 5. Check coverage before reporting results
Smaller MAE/RMSE is better. Correlation measures whether ups and downs track each other, and does not measure error size. Report the actual test end date and evaluated cells. Do not compare these numbers directly with a teammate's longer test period or same-month nowcast.

The figures and tables above are generated only when the real input is present. Synthetic test results are not evidence of scientific performance.

In [ ]:
import json
manifest = json.loads((OUTPUT / "run_manifest.json").read_text())
display(manifest)
display(pd.read_csv(OUTPUT / "split_manifest.csv").query("split == 'test'"))

## 6. Notes to write after running
- Which relationships remain after seasonal adjustment?
- How do the distributions differ between training and testing?
- Does Ridge improve on both persistence and seasonal persistence, on the common mask?
- Which months/locations were excluded and why?
- What limitations should the group mention?

Write findings from the actual tables. An individual literature review still needs to be completed separately.